## Data Preparation and Tokenization

Welcome back to Bringing Transformers to Life: Training & Inference! Excellent progress so far. In our first lesson, you successfully assembled a complete Transformer model by integrating all the architectural components we've built throughout this learning journey. You now have a powerful, production-ready model capable of handling sequence-to-sequence tasks.

However, even the most sophisticated model is only as good as the data it receives. In this second lesson, we shift our focus to a foundational yet critical aspect: data preparation and tokenization. While our Transformer model is ready to process sequences, we need to bridge the gap between raw text and the numerical tensors that neural networks understand. This lesson will guide you through building a complete data preprocessing pipeline, from raw sentences to properly batched, padded tensors ready for training. We'll explore vocabulary building, special token handling, and efficient data loading strategies that ensure your Transformer receives well-structured input for optimal learning.

---

### Understanding the Sequence-to-Sequence Data Pipeline

Before diving into implementation, let's establish the complete data pipeline for sequence-to-sequence learning. Unlike simple classification tasks, seq2seq models require careful coordination between source and target sequences, each potentially having different vocabularies and lengths.

The pipeline consists of several interconnected stages: tokenization converts raw text into discrete tokens (words or characters), vocabulary building creates mappings between tokens and numerical indices, special token integration handles sequence boundaries and unknown words, and dynamic batching efficiently groups variable-length sequences. Each stage must handle the unique challenges of seq2seq tasks, such as different source and target languages, varying sentence lengths within batches, and the need for teacher forcing during training. Understanding this pipeline is crucial because any inefficiency or error propagates through training, potentially degrading model performance or causing training instability.

---

### Building a Robust Vocabulary System

Our vocabulary system forms the backbone of text-to-tensor conversion. Let's examine how we create a robust vocabulary that handles both known and unknown tokens:

```python
class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)
```

This vocabulary design incorporates four essential special tokens that enable robust sequence handling. The `<PAD>` token (index 0) allows batching sequences of different lengths, `<SOS>` marks sequence starts for decoder initialization, `<EOS>` signals sequence completion, and `<UNK>` gracefully handles out-of-vocabulary words. The `build_vocab` method uses frequency-based filtering through `min_freq`, preventing rare tokens that might cause overfitting while maintaining vocabulary manageability. Notice how we maintain bidirectional mappings: `token2idx` for encoding text into numbers and `idx2token` for decoding numbers back to text.

Let's complete our vocabulary with the encoding and decoding methods:

```python
    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()
```

The `encode` method demonstrates graceful unknown token handling using the `get` method with `<UNK>` as the default. This ensures that even completely unseen words don't break the pipeline, instead being mapped to a special unknown token that the model can learn to handle appropriately. The `decode` method performs the inverse transformation while intelligently filtering out padding tokens and cleaning up special tokens that shouldn't appear in the final output. This asymmetric processing (adding special tokens during encoding, removing them during decoding) is fundamental to seq2seq training and inference workflows.

---

### Creating the Translation Dataset

To demonstrate our pipeline, we'll create synthetic translation data and a PyTorch dataset class. We consider a very simple translation task: given a string consisting of multiple words, return another string in which the word order is reversed; for example, given the string "how are you", the model should return "you are how". While this task may seem trivial, it allows us to keep the training process complexity within reasonable time boundaries, while also building a scaffolding that can easily scale to real-world tasks (such as language translation) by simply changing the data source. In other words, this can be seen as the "Hello World" for a Transformer!

As first step, let's generate the data:

```python
def create_synthetic_data(num_samples=1000, seed=42):
    """Create synthetic translation data (English to reversed English)"""
    random.seed(seed)
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences
```

This synthetic data generator creates a controlled learning environment perfect for understanding data pipeline mechanics. By reversing word order ("hello world" becomes "world hello"), we create a deterministic translation task that's complex enough to require the full seq2seq architecture yet simple enough to debug and verify.

---

### Implementing Translation Dataset with Teacher Forcing

Now let's implement the dataset class:

```python
class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }
```

Notice how the target sequence handling prepares for a training technique called teacher forcing: `tgt_indices` starts with `<SOS>` for the decoder input, while `tgt_output` ends with `<EOS>` for loss computation. This offset pattern is essential for how the model will be trained, allowing the decoder to receive the correct previous token as input while learning to predict the next token. While you don’t need to fully understand teacher forcing yet, know that this setup is standard for sequence-to-sequence training and will be explained in detail in the next lesson. The dataset returns three tensors per sample, creating the essential triplet structure needed for efficient seq2seq training.

---

### Implementing Dynamic Batching

Efficient batching requires dynamic padding since sequences within a batch typically have different lengths. Let's implement a custom collate function:

```python
def collate_fn(batch):
    """Custom collate function for dynamic padding"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    # Pad sequences to max length in batch
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)

    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }
```

This custom collate function implements adaptive padding that only pads to the maximum length within each batch, not across the entire dataset. This strategy significantly reduces computational waste compared to global padding while maintaining tensor uniformity required for efficient GPU processing. The `pad_sequence` function automatically handles the padding logic, using our designated padding value (0, corresponding to the `<PAD>` token). The `batch_first=True` parameter ensures compatibility with our Transformer implementation's expected input format.

---

### Validating the Complete Pipeline

Let's execute our complete pipeline and examine the outputs to verify correct functionality:

```python
def main():
    """Test data preparation pipeline"""
    # Create synthetic data
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    print(f"Created {len(src_sentences)} sentence pairs")
    print("Sample data:")
    for i in range(3):
        print(f"  Source: {src_sentences[i]}")
        print(f"  Target: {tgt_sentences[i]}")

    # Build vocabularies
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()

    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    print(f"Source vocabulary size: {src_vocab.size}")
    print(f"Target vocabulary size: {tgt_vocab.size}")
```

The initial output demonstrates our synthetic data generation and vocabulary building:

```text
Created 500 sentence pairs
Sample data:
  Source: see you later
  Target: later you see
  Source: have a nice day
  Target: day nice a have
  Source: thank you very much
  Target: much very you thank
Source vocabulary size: 22
Target vocabulary size: 22
```

Both vocabularies reach identical sizes (22 tokens) because our synthetic task uses the same words in both source and target, just in different orders. Let's complete the validation by testing batch processing:

```python
    # Create dataset and dataloader
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

    # Test one batch
    batch = next(iter(dataloader))

    print(f"Batch source shape: {batch['src'].shape}")
    print(f"Batch target shape: {batch['tgt'].shape}")
    print(f"Batch target output shape: {batch['tgt_output'].shape}")

    # Decode sample
    sample_src = batch['src'][0]
    sample_tgt = batch['tgt_output'][0]

    decoded_src = src_vocab.decode(sample_src.tolist())
    decoded_tgt = tgt_vocab.decode(sample_tgt.tolist())

    print(f"Decoded source: '{decoded_src}'")
    print(f"Decoded target: '{decoded_tgt}'")
```

The final output confirms successful tensor creation and round-trip encoding/decoding:

```text
Batch source shape: torch.Size([4, 4])
Batch target shape: torch.Size([4, 5])
Batch target output shape: torch.Size([4, 5])
Decoded source: 'nice to meet you'
Decoded target: 'you meet to nice'
```

Notice how the target sequences are longer (5 tokens) than the source sequences (4 tokens) due to the added `<SOS>` and `<EOS>` tokens. The successful decoding demonstrates that our vocabulary system correctly handles the complete encode-decode cycle, preserving semantic content while managing special tokens appropriately.

---

### Conclusion and Next Steps

You've successfully built a comprehensive data preparation pipeline that transforms raw text into training-ready tensors for sequence-to-sequence learning. This pipeline elegantly handles vocabulary construction, special token management, dynamic padding, and efficient batching — all critical components for successful Transformer training. Your implementation demonstrates a sophisticated understanding of the challenges inherent in seq2seq data processing, from handling variable-length sequences to implementing teacher forcing preparation.

The robust foundation you've established will serve as the cornerstone for all subsequent training and inference work. In the upcoming practice exercises, you'll apply this pipeline to more complex scenarios and explore how different tokenization strategies impact model performance, deepening your expertise in the crucial intersection between raw data and neural network training.

## Building Your First Vocabulary System

Now that you've learned the fundamentals of data preparation and tokenization, it's time to put that knowledge into practice by building a complete vocabulary system.

Your goal is to implement all the missing methods in the `Vocabulary` class. This class will handle the core functionality needed for any sequence-to-sequence model.

---

### Your Task

- Initialize the vocabulary with special tokens (`<PAD>`, `<SOS>`, `<EOS>`, `<UNK>`).
- Add new tokens to the vocabulary dynamically.
- Build the vocabulary from sentences with frequency filtering.
- Convert sentences to token indices (encoding).
- Convert token indices back to sentences (decoding).

Once you complete this implementation, you'll have a fully functional vocabulary system that can handle any text data for training!

---

### Starter Code

```python
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        # TODO: Initialize the token2idx dictionary with special tokens
        # Special tokens should be: <PAD>=0, <SOS>=1, <EOS>=2, <UNK>=3
        pass

    def add_token(self, token):
        # TODO: Add a new token to the vocabulary if it doesn't exist
        # Update both token2idx and idx2token mappings
        # Don't forget to update the size
        pass

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        # TODO: Count word frequencies using Counter
        # TODO: Add tokens that appear at least min_freq times
        pass

    def encode(self, sentence):
        """Convert sentence to token indices"""
        # TODO: Split the sentence into tokens
        # TODO: Convert each token to its index, use <UNK> for unknown tokens
        pass

    def decode(self, indices):
        """Convert token indices back to sentence"""
        # TODO: Convert indices back to tokens, skip <PAD> tokens
        # TODO: Join tokens into a sentence and clean up special tokens
        pass

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences
```

---

### Solution

Every TODO is inside the `Vocabulary` class:

```python
def __init__(self):
    self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
    self.size = 4

def add_token(self, token):
    if token not in self.token2idx:
        self.token2idx[token] = self.size
        self.idx2token[self.size] = token
        self.size += 1

def build_vocab(self, sentences, min_freq=1):
    counter = Counter()
    for sentence in sentences:
        counter.update(sentence.split())

    for token, freq in counter.items():
        if freq >= min_freq:
            self.add_token(token)

def encode(self, sentence):
    tokens = sentence.split()
    return [self.token2idx.get(token, self.token2idx['<UNK>'])
            for token in tokens]

def decode(self, indices):
    tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
    return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()
```

- `__init__` reserves indices 0 to 3 for the special tokens, and `size = 4` makes the next new token get index 4. `idx2token` is the reverse mapping, used for decoding.
- `add_token` is idempotent: a known token is skipped, so `size` always equals the number of entries and the two dictionaries stay in sync.
- `build_vocab` counts words with `Counter` and adds only tokens seen at least `min_freq` times, so rare words can be left out of the vocabulary.
- `encode` uses `dict.get` with `<UNK>` as the default, so an unseen word maps to index 3 instead of raising a `KeyError`.
- `decode` drops `<PAD>` while converting, then strips `<SOS>` and `<EOS>` from the joined string.
- Without the fix, every method returns `None` because of `pass`: `__init__` creates no dictionaries, so `TranslationDataset.__getitem__` would fail at `self.src_vocab.encode(...)` with a `TypeError` (`None` cannot be iterated).

The starter code has no `main()`, so the code cell below adds a labelled quick check at the end that builds both vocabularies from the synthetic data and round-trips a few sentences. It is not part of the starter code.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        # Special tokens: <PAD>=0, <SOS>=1, <EOS>=2, <UNK>=3
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4
    
    def add_token(self, token):
        # Add the token only if it is new, keeping both mappings and size in sync
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1
    
    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        # Count word frequencies, then keep tokens that appear at least min_freq times
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())
        
        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)
    
    def encode(self, sentence):
        """Convert sentence to token indices"""
        # Split into tokens and map each one to its index, falling back to <UNK>
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]
            
    def decode(self, indices):
        """Convert token indices back to sentence"""
        # Skip <PAD>, join the tokens and strip <SOS>/<EOS>
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len
    
    def __len__(self):
        return len(self.src_sentences)
    
    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]
        
        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence) 
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')
        
        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]
    
    src_sentences = []
    tgt_sentences = []
    
    for _ in range(num_samples):
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))
    
    return src_sentences, tgt_sentences


# Quick check (not part of the starter code): build vocabularies and round-trip a sentence
def main():
    random.seed(42)
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    print(f"Source vocabulary size: {src_vocab.size}")
    print(f"Target vocabulary size: {tgt_vocab.size}")

    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
    sample = dataset[0]
    print(f"Source: '{src_sentences[0]}' -> {sample['src'].tolist()}")
    print(f"Target input:  {sample['tgt'].tolist()} -> '{tgt_vocab.decode(sample['tgt'].tolist())}'")
    print(f"Target output: {sample['tgt_output'].tolist()} -> '{tgt_vocab.decode(sample['tgt_output'].tolist())}'")
    print(f"Unknown word: {src_vocab.encode('hello stranger')}")
    print(f"min_freq=1000 vocabulary size: {(lambda v: (v.build_vocab(src_sentences, min_freq=1000), v.size)[1])(Vocabulary())}")

if __name__ == "__main__":
    main()


## Building Your Translation Dataset

Now that you've built a solid `Vocabulary` class, it's time to create the dataset that will feed data to your model. You'll implement the `TranslationDataset` class, which is responsible for loading and preparing sentence pairs for training.

---

### Your Task

Complete the three core methods:

- Store the input data and vocabularies in the constructor.
- Return the dataset size when requested.
- Convert sentences to token indices and package them properly for each sample.

The `__getitem__` method is the most important part — it needs to encode both source and target sentences using the appropriate vocabularies and create three versions of the target: one with `<SOS>` at the start for decoder input and one with `<EOS>` at the end for training targets.

This dataset class will be the bridge between your raw text data and your model, so getting it right is key to successful training!

---

### Starter Code

```python
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        # TODO: Store the sentences and vocabularies as instance variables
        pass

    def __len__(self):
        # TODO: Return the number of sentence pairs in the dataset
        pass

    def __getitem__(self, idx):
        # TODO: Get the source and target sentences at the given index

        # TODO: Encode the source sentence using src_vocab

        # TODO: Encode the target sentence with <SOS> token at the beginning using tgt_vocab

        # TODO: Encode the target sentence with <EOS> token at the end using tgt_vocab

        # TODO: Return a dictionary with 'src', 'tgt', and 'tgt_output' keys
        # Each value should be a PyTorch tensor of type long
        pass

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences
```

---

### Solution

Every TODO is inside the `TranslationDataset` class:

```python
def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
    self.src_sentences = src_sentences
    self.tgt_sentences = tgt_sentences
    self.src_vocab = src_vocab
    self.tgt_vocab = tgt_vocab
    self.max_len = max_len

def __len__(self):
    return len(self.src_sentences)

def __getitem__(self, idx):
    src_sentence = self.src_sentences[idx]
    tgt_sentence = self.tgt_sentences[idx]

    src_indices = self.src_vocab.encode(src_sentence)
    tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
    tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

    return {
        'src': torch.tensor(src_indices, dtype=torch.long),
        'tgt': torch.tensor(tgt_indices, dtype=torch.long),
        'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
    }
```

- `__init__` keeps the sentences, both vocabularies and `max_len` on `self`, so `__len__` and `__getitem__` can use them.
- `__len__` returns the number of sentence pairs, which `DataLoader` needs to know how many samples exist.
- `__getitem__` encodes the source with `src_vocab` and the target with `tgt_vocab`. The two target versions are shifted by one token (teacher forcing): `tgt` starts with `<SOS>` and is the decoder input, and `tgt_output` ends with `<EOS>` and is what the decoder should predict at each step.
- The tensors use `dtype=torch.long` because they are indices for `nn.Embedding` and the loss.
- The task text says "three versions of the target", but only two are listed and returned. With the `src` tensor, the sample holds three tensors: `src`, `tgt` and `tgt_output`.
- Without the fix, `__init__` sets nothing, `__len__` returns `None` (so `len(dataset)` raises a `TypeError`) and `__getitem__` returns `None`, so the batch can't be built.

The starter code has no `main()`, so the code cell below adds a labelled quick check at the end that builds a dataset from the synthetic data and prints one sample. It is not part of the starter code.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        # Store the sentences and vocabularies as instance variables
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        # One sample per sentence pair
        return len(self.src_sentences)

    def __getitem__(self, idx):
        # Get the source and target sentences at the given index
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode the source sentence using src_vocab
        src_indices = self.src_vocab.encode(src_sentence)

        # Encode the target sentence with <SOS> at the beginning (decoder input)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)

        # Encode the target sentence with <EOS> at the end (training target)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        # Return a dictionary of long tensors
        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences


# Quick check (not part of the starter code): build a dataset and inspect one sample
def main():
    random.seed(42)
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
    print(f"Dataset size: {len(dataset)}")

    sample = dataset[0]
    print(f"Source sentence: '{src_sentences[0]}'")
    print(f"Target sentence: '{tgt_sentences[0]}'")
    for key, value in sample.items():
        print(f"{key}: {value.tolist()} (dtype={value.dtype})")

if __name__ == "__main__":
    main()


## Building Your Translation Dataset

Now that you've built the `Vocabulary` and `TranslationDataset` classes, it's time to create some training data to work with. In this exercise, you'll complete the `create_synthetic_data` function, which generates pairs of sentences for a simple translation scenario.

---

### Your Task

The function should create synthetic translation data in which the target sentence is simply the source sentence with the words in reverse order. For example, "hello world" becomes "world hello". You'll need to randomly select templates from the provided list and create the corresponding reversed versions.

- Inside the loop, pick a random template.
- Add it to the source sentences list.
- Create the reversed version and add it to the target sentences list.

This synthetic data will provide a perfect dataset to test your data pipeline!

---

### Starter Code

```python
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        # TODO: Randomly choose a template and create source/target pair
        pass

    return src_sentences, tgt_sentences
```

---

### Solution

The TODO is the loop body in `create_synthetic_data`:

```python
for _ in range(num_samples):
    template = random.choice(templates)
    src_sentences.append(template)
    tgt_sentences.append(" ".join(template.split()[::-1]))
```

- `random.choice(templates)` picks one of the seven sentences at random, so each pair is drawn independently.
- The source is the template itself.
- `template.split()[::-1]` splits it into words and reverses the list, and `" ".join(...)` turns it back into a string. `"how are you"` becomes `"you are how"`.
- Both lists are appended once per iteration, so they stay aligned: `src_sentences[i]` always pairs with `tgt_sentences[i]`.
- Without the fix, the loop body is only `pass`. Both lists stay empty and the function returns two empty lists, so no training pairs are created and the vocabulary and dataset built on top of them are empty.

The starter code has no `main()`, so the code cell below adds a labelled quick check at the end that generates the data, prints three pairs and checks that every target is the reversed source. It is not part of the starter code. The function takes no seed, so the quick check calls `random.seed(42)` first to make the output repeatable.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        # Pick a random template; the target is the same words in reverse order
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences


# Quick check (not part of the starter code): generate data and inspect a few pairs
def main():
    random.seed(42)
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    print(f"Created {len(src_sentences)} sentence pairs")
    print("Sample data:")
    for i in range(3):
        print(f"  Source: {src_sentences[i]}")
        print(f"  Target: {tgt_sentences[i]}")

    assert all(t == " ".join(s.split()[::-1]) for s, t in zip(src_sentences, tgt_sentences))
    print("All targets are the reversed sources")

if __name__ == "__main__":
    main()


## Dynamic Padding for Sequence Batches

Now that you have synthetic data ready, it's time to handle an important challenge in batch processing: sequences of different lengths. Your job is to implement the `collate_fn` function, which handles dynamic padding for efficient batching.

---

### Your Task

The provided code already extracts the source, target, and target output sequences from each batch item. However, these sequences have different lengths, which causes issues when trying to create tensors. You need to add the padding logic that makes all sequences in a batch the same length by adding padding tokens where needed.

- Use PyTorch's built-in padding function to pad each sequence type to the maximum length in the batch.
- This will allow the `DataLoader` to create properly shaped tensors for training.

Once you implement this correctly, your data pipeline will be ready to handle variable-length sequences efficiently!

---

### Starter Code

```python
import torch
from torch.utils.data import DataLoader

from data_utils import Vocabulary, TranslationDataset, create_synthetic_data

def collate_fn(batch):
    """Custom collate function for dynamic padding"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    # TODO: Pad sequences to max length in batch using torch.nn.utils.rnn.pad_sequence

    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }

def main():
    """Test data preparation pipeline"""
    # Create synthetic data
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    print(f"Created {len(src_sentences)} sentence pairs")
    print("Sample data:")
    for i in range(3):
        print(f"  Source: {src_sentences[i]}")
        print(f"  Target: {tgt_sentences[i]}")

    # Build vocabularies
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()

    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    print(f"Source vocabulary size: {src_vocab.size}")
    print(f"Target vocabulary size: {tgt_vocab.size}")

    # Create dataset and dataloader
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

    # Test one batch
    batch = next(iter(dataloader))

    print(f"Batch source shape: {batch['src'].shape}")
    print(f"Batch target shape: {batch['tgt'].shape}")
    print(f"Batch target output shape: {batch['tgt_output'].shape}")

    # Decode sample
    sample_src = batch['src'][0]
    sample_tgt = batch['tgt_output'][0]

    decoded_src = src_vocab.decode(sample_src.tolist())
    decoded_tgt = tgt_vocab.decode(sample_tgt.tolist())

    print(f"Decoded source: '{decoded_src}'")
    print(f"Decoded target: '{decoded_tgt}'")


if __name__ == "__main__":
    main()
```

---

### Solution

The TODO is inside `collate_fn`:

```python
from torch.nn.utils.rnn import pad_sequence

src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0)
tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
tgt_output_batch = pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
```

- `pad_sequence` takes a list of 1-D tensors of different lengths and stacks them into one tensor, filling the shorter ones up to the longest length in the batch. That is why the padding is "dynamic": each batch is padded only as much as it needs.
- `batch_first=True` makes the result `(batch, seq_len)`. Without it, the shape would be `(seq_len, batch)`.
- `padding_value=0` is the index of `<PAD>` in `Vocabulary`, so padded positions are ignored by `decode` and can be masked later.
- It is applied to all three lists (`src`, `tgt`, `tgt_output`), since each has its own lengths.
- Without the fix, `collate_fn` returns plain Python lists of tensors, so `batch['src'].shape` raises an `AttributeError` and the `DataLoader` cannot give the model a single tensor per batch.

The starter code imports `Vocabulary`, `TranslationDataset` and `create_synthetic_data` from CodeSignal's `data_utils` module, which does not exist in this notebook. The code cell below uses the inline versions built in the earlier exercises and adds the `pad_sequence` import. The data is shuffled and not seeded, so the sample and the exact batch shapes differ from run to run.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        # Pick a random template; the target is the same words in reverse order
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences


def collate_fn(batch):
    """Custom collate function for dynamic padding"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    # Pad sequences to max length in batch using torch.nn.utils.rnn.pad_sequence
    # batch_first=True gives (batch, seq_len); padding_value=0 is the <PAD> index
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)

    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }

def main():
    """Test data preparation pipeline"""
    # Create synthetic data
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    print(f"Created {len(src_sentences)} sentence pairs")
    print("Sample data:")
    for i in range(3):
        print(f"  Source: {src_sentences[i]}")
        print(f"  Target: {tgt_sentences[i]}")

    # Build vocabularies
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()

    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    print(f"Source vocabulary size: {src_vocab.size}")
    print(f"Target vocabulary size: {tgt_vocab.size}")

    # Create dataset and dataloader
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

    # Test one batch
    batch = next(iter(dataloader))

    print(f"Batch source shape: {batch['src'].shape}")
    print(f"Batch target shape: {batch['tgt'].shape}")
    print(f"Batch target output shape: {batch['tgt_output'].shape}")

    # Decode sample
    sample_src = batch['src'][0]
    sample_tgt = batch['tgt_output'][0]

    decoded_src = src_vocab.decode(sample_src.tolist())
    decoded_tgt = tgt_vocab.decode(sample_tgt.tolist())

    print(f"Decoded source: '{decoded_src}'")
    print(f"Decoded target: '{decoded_tgt}'")


if __name__ == "__main__":
    main()


## Building Your Complete Data Pipeline

Now that you've built the collate function for dynamic padding, it's time to put all the pieces together and create a complete data preparation pipeline! Your job is to implement the entire `main` function from scratch, bringing together all the components you've built in previous exercises.

---

### Your Task

The provided code contains a skeleton `main` function with helpful comments indicating what needs to be done. You'll need to:

- Generate synthetic translation data and display sample pairs.
- Build vocabularies for both the source and target languages.
- Create a dataset and dataloader with proper batching.
- Test the pipeline by processing a batch and showing the results.

This is your chance to see the full data preparation pipeline in action and verify that everything works together smoothly!

---

### Starter Code

```python
import torch
from torch.utils.data import DataLoader

from data_utils import Vocabulary, TranslationDataset, create_synthetic_data

def collate_fn(batch):
    """Custom collate function for dynamic padding"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    # Pad sequences to max length in batch
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)

    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }

def main():
    """Test data preparation pipeline"""
    # TODO: Create synthetic data with 500 samples

    # TODO: Print the number of sentence pairs created

    # TODO: Print sample data - show first 3 source and target pairs

    # TODO: Build vocabularies for source and target sentences

    # TODO: Print vocabulary sizes

    # TODO: Create dataset and dataloader with batch size 4

    # TODO: Get the first batch from the dataloader

    # TODO: Print the shapes of source, target, and target output tensors

    # TODO: Decode the first sample in the batch and print the results


if __name__ == "__main__":
    main()
```

---

### Solution

All nine TODOs are inside `main`:

```python
src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)
print(f"Created {len(src_sentences)} sentence pairs")

print("Sample data:")
for i in range(3):
    print(f"  Source: {src_sentences[i]}")
    print(f"  Target: {tgt_sentences[i]}")

src_vocab = Vocabulary()
tgt_vocab = Vocabulary()
src_vocab.build_vocab(src_sentences)
tgt_vocab.build_vocab(tgt_sentences)
print(f"Source vocabulary size: {src_vocab.size}")
print(f"Target vocabulary size: {tgt_vocab.size}")

dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

batch = next(iter(dataloader))
print(f"Batch source shape: {batch['src'].shape}")
print(f"Batch target shape: {batch['tgt'].shape}")
print(f"Batch target output shape: {batch['tgt_output'].shape}")

decoded_src = src_vocab.decode(batch['src'][0].tolist())
decoded_tgt = tgt_vocab.decode(batch['tgt_output'][0].tolist())
print(f"Decoded source: '{decoded_src}'")
print(f"Decoded target: '{decoded_tgt}'")
```

- `create_synthetic_data` gives the sentence pairs. The printed count and the first three pairs confirm it worked.
- Each language gets its own `Vocabulary`, built from its own sentences, so `src_vocab` and `tgt_vocab` are separate objects.
- `TranslationDataset` encodes each pair, and `DataLoader` with `collate_fn=collate_fn` pads each batch of 4 to its longest sequence.
- `next(iter(dataloader))` takes one batch, so the three shapes are `(4, seq_len)`. The target and target output have the same length, because each has one extra token (`<SOS>` or `<EOS>`).
- The decoded source and target use `src_vocab.decode` and `tgt_vocab.decode` on the first row. The target uses `tgt_output`, which ends with `<EOS>`, and `decode` strips it. The decoded target is the reversed source.
- Without the fix, `main` does nothing: the body is only comments, so no data is created and nothing is printed.

The starter code imports `Vocabulary`, `TranslationDataset` and `create_synthetic_data` from CodeSignal's `data_utils` module, which does not exist in this notebook. The code cell below uses the inline versions built in the earlier exercises. The data is shuffled and not seeded, so the first batch differs from run to run.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

class Vocabulary:
    def __init__(self):
        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.size = 4

    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = self.size
            self.idx2token[self.size] = token
            self.size += 1

    def build_vocab(self, sentences, min_freq=1):
        """Build vocabulary from list of sentences"""
        counter = Counter()
        for sentence in sentences:
            counter.update(sentence.split())

        for token, freq in counter.items():
            if freq >= min_freq:
                self.add_token(token)

    def encode(self, sentence):
        """Convert sentence to token indices"""
        tokens = sentence.split()
        return [self.token2idx.get(token, self.token2idx['<UNK>'])
                for token in tokens]

    def decode(self, indices):
        """Convert token indices back to sentence"""
        tokens = [self.idx2token[idx] for idx in indices if idx != self.token2idx['<PAD>']]
        return ' '.join(tokens).replace('<SOS>', '').replace('<EOS>', '').strip()

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx]
        tgt_sentence = self.tgt_sentences[idx]

        # Encode sentences
        src_indices = self.src_vocab.encode(src_sentence)
        tgt_indices = self.tgt_vocab.encode('<SOS> ' + tgt_sentence)
        tgt_output = self.tgt_vocab.encode(tgt_sentence + ' <EOS>')

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long),
            'tgt_output': torch.tensor(tgt_output, dtype=torch.long)
        }

def create_synthetic_data(num_samples=1000):
    """Create synthetic translation data (English to reversed English)"""
    templates = [
        "hello world", "good morning", "how are you", "nice to meet you",
        "thank you very much", "see you later", "have a nice day"
    ]

    src_sentences = []
    tgt_sentences = []

    for _ in range(num_samples):
        # Pick a random template; the target is the same words in reverse order
        template = random.choice(templates)
        src_sentences.append(template)
        tgt_sentences.append(" ".join(template.split()[::-1]))

    return src_sentences, tgt_sentences


def collate_fn(batch):
    """Custom collate function for dynamic padding"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]

    # Pad sequences to max length in batch
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)

    return {
        'src': src_batch,
        'tgt': tgt_batch,
        'tgt_output': tgt_output_batch
    }


def main():
    """Test data preparation pipeline"""
    # Create synthetic data with 500 samples
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=500)

    # Print the number of sentence pairs created
    print(f"Created {len(src_sentences)} sentence pairs")

    # Print sample data - show first 3 source and target pairs
    print("Sample data:")
    for i in range(3):
        print(f"  Source: {src_sentences[i]}")
        print(f"  Target: {tgt_sentences[i]}")

    # Build vocabularies for source and target sentences
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    # Print vocabulary sizes
    print(f"Source vocabulary size: {src_vocab.size}")
    print(f"Target vocabulary size: {tgt_vocab.size}")

    # Create dataset and dataloader with batch size 4
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

    # Get the first batch from the dataloader
    batch = next(iter(dataloader))

    # Print the shapes of source, target, and target output tensors
    print(f"Batch source shape: {batch['src'].shape}")
    print(f"Batch target shape: {batch['tgt'].shape}")
    print(f"Batch target output shape: {batch['tgt_output'].shape}")

    # Decode the first sample in the batch and print the results
    decoded_src = src_vocab.decode(batch['src'][0].tolist())
    decoded_tgt = tgt_vocab.decode(batch['tgt_output'][0].tolist())
    print(f"Decoded source: '{decoded_src}'")
    print(f"Decoded target: '{decoded_tgt}'")


if __name__ == "__main__":
    main()
